<a href="https://colab.research.google.com/github/vzavala/StatsBook/blob/master/Python/ch6_gibbs_neuralnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Setup.** Run this cell first: it locates the data files this notebook reads. On Google
Colab it downloads them (the `Data/` and `Support/` folders) from the book's GitHub
repository; on your computer, run the notebook from the folder that contains `Data/`.

In [1]:
import os
import sys
path = ''
if 'google.colab' in sys.modules and not os.path.isdir('Data'):
    if not os.path.isdir('statsbook'):
        os.system('git clone -q --depth 1 --filter=blob:none --sparse '
                  'https://github.com/vzavala/StatsBook.git statsbook')
        os.system('git -C statsbook sparse-checkout set Python/Data Python/Support')
    path = 'statsbook/Python/'

Neural net versus kriging for the Gibbs reactor (Victor Z and Ugochukwu Ikegwu, UW-Madison, 2024).
Inputs $x_\omega=(P,T)$ (`X`, scaled), output conversion $y_\omega$ (`y`). The NN has one hidden layer
with two perceptrons (logistic activations) and is trained by minimizing the squared loss $L(\theta)$.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler

data = np.vstack((np.loadtxt(path+'Data/gibbs_hightemp_class.dat'),
                  np.loadtxt(path+'Data/gibbs_lowtemp_class.dat')))
S = len(data)
np.random.seed(0)
X = np.column_stack((data[:, 1], data[:, 0])) + np.random.rand(S, 2) * 0.1   # (P, T)
y = data[:, 2] + np.random.rand(S) * 0.1
X = StandardScaler().fit_transform(X)

nn = MLPRegressor(hidden_layer_sizes=(2,), activation='logistic', solver='lbfgs',
                  max_iter=20000, random_state=2024).fit(X, y)
y_nn = nn.predict(X)
gp = GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=1.5) + WhiteKernel(0.01), normalize_y=True,
                              n_restarts_optimizer=5, random_state=0).fit(X, y)
y_gp = gp.predict(X)
n_params = sum(w.size for w in nn.coefs_) + sum(b.size for b in nn.intercepts_)
print(f"NN parameters: {n_params};  MSE NN = {np.mean((y - y_nn)**2):.2e},  MSE kriging = {np.mean((y - y_gp)**2):.2e}")

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, pred, title in ((axs[0], y_gp, 'kriging'), (axs[1], y_nn, 'neural net (2 hidden perceptrons)')):
    ax.plot(y, pred, 'ko', markerfacecolor='w', markersize=3)
    ax.plot([0, 1.1], [0, 1.1], 'k-')
    ax.set_title(title); ax.set_xlabel(r'$y_\omega$'); ax.set_ylabel(r'$\hat{y}_\omega$')
    ax.axis([0, 1.1, 0, 1.1]); ax.grid(True)
plt.tight_layout()
plt.show()

NN parameters: 9;  MSE NN = 9.29e-04,  MSE kriging = 8.71e-04


/private/tmp/claude-501/-Users-vzavala-ClaudeProjects-Statsbook/a512c131-6094-4621-8705-579e14a50fbc/scratchpad/venv/lib/python3.9/site-packages/sklearn/gaussian_process/_gpr.py:441: RuntimeWarning: divide by zero encountered in matmul
  y_mean = K_trans @ self.alpha_
/private/tmp/claude-501/-Users-vzavala-ClaudeProjects-Statsbook/a512c131-6094-4621-8705-579e14a50fbc/scratchpad/venv/lib/python3.9/site-packages/sklearn/gaussian_process/_gpr.py:441: RuntimeWarning: overflow encountered in matmul
  y_mean = K_trans @ self.alpha_
/private/tmp/claude-501/-Users-vzavala-ClaudeProjects-Statsbook/a512c131-6094-4621-8705-579e14a50fbc/scratchpad/venv/lib/python3.9/site-packages/sklearn/gaussian_process/_gpr.py:441: RuntimeWarning: invalid value encountered in matmul
  y_mean = K_trans @ self.alpha_
/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40791/3548271532.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
